[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-02-connecting-data-sources.ipynb#scrollTo=aa110001)

---
# Day 2 · Connecting to Data Sources — DuckDB, PostgreSQL, and Snowflake
**certified-journeys / sodacore-certified** · Day 2 · Configuration

> **Goal for today:** Write correct `configuration.yml` files for DuckDB, PostgreSQL, and Snowflake, use environment variable substitution to keep credentials out of version control, and run a connection-validation scan equivalent to `soda test-connection`.

In [ ]:
%pip install -q soda-core-duckdb

## Step 1 · The Soda Data Source Configuration Reference

Every Soda scan needs a `configuration.yml` that maps **data source names** to connection details. You can define multiple sources in one file:

```yaml
data_sources:
  source_name_1:
    type: <connector_type>
    # ... connector-specific keys
  source_name_2:
    type: <connector_type>
    # ...
```

**Supported connectors (install package → type key):**

| Connector package | `type` value | Notes |
|-------------------|-------------|-------|
| `soda-core-duckdb` | `duckdb` | Local/embedded, great for dev |
| `soda-core-postgres` | `postgres` | PostgreSQL + compatible DBs |
| `soda-core-snowflake` | `snowflake` | Snowflake warehouse |
| `soda-core-bigquery` | `bigquery` | Google BigQuery |
| `soda-core-spark-df` | `spark_df` | Apache Spark DataFrames |
| `soda-core-mysql` | `mysql` | MySQL / MariaDB |
| `soda-core-redshift` | `redshift` | Amazon Redshift |
| `soda-core-trino` | `trino` | Trino / Presto |

The **`type` value** is fixed per connector — it must match exactly what the connector registers with Soda.

In [ ]:
import yaml

# Build the full connector reference as a Python dict for exploration
connectors = [
    {"package": "soda-core-duckdb",    "type": "duckdb",    "required_keys": ["path"]},
    {"package": "soda-core-postgres",  "type": "postgres",  "required_keys": ["host", "port", "database", "username", "password"]},
    {"package": "soda-core-snowflake", "type": "snowflake", "required_keys": ["username", "password", "account", "database", "warehouse"]},
    {"package": "soda-core-bigquery",  "type": "bigquery",  "required_keys": ["project_id", "dataset"]},
    {"package": "soda-core-redshift",  "type": "redshift",  "required_keys": ["host", "port", "database", "username", "password"]},
]

print(f"{'Package':<30} {'Type':<12} {'Required keys'}")
print("-" * 70)
for c in connectors:
    print(f"{c['package']:<30} {c['type']:<12} {', '.join(c['required_keys'])}")

### What just happened?
- Each connector requires its own pip package — `soda-core-duckdb` does NOT include the Postgres connector.
- The `type` key is the critical link between configuration and the installed connector package.
- **`path`** is DuckDB's only required key — for file-based databases you just need a file path.
- For cloud warehouses (Snowflake, BigQuery), credentials should always come from environment variables, not hardcoded values.

## Step 2 · Configuring DuckDB — Full Working Example

DuckDB is the best connector for learning and local testing because it requires no server. The full configuration reference:

```yaml
data_sources:
  my_duckdb:
    type: duckdb
    path: /path/to/database.duckdb   # required — absolute path
    # Optional:
    # schema: main                   # default schema (DuckDB default is 'main')
```

**Important DuckDB notes:**
- Use a file path — Soda cannot use `:memory:` because it opens its own connection.
- Close your own DuckDB connection before running a Soda scan on the same file.
- The default schema in DuckDB is `main` — most tables created without a schema prefix live there.

In [ ]:
import duckdb
import tempfile
import pathlib
from soda.scan import Scan

# --- Setup: create a temp DuckDB file with sample data ---
tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "analytics.duckdb")

conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE events AS
    SELECT * FROM (VALUES
        (1,  'page_view',  'user_001', '2024-01-01 08:00:00'),
        (2,  'click',      'user_002', '2024-01-01 08:05:00'),
        (3,  'signup',     'user_003', '2024-01-01 08:10:00'),
        (4,  'page_view',  'user_001', '2024-01-01 08:15:00'),
        (5,  'purchase',   'user_002', '2024-01-01 08:20:00'),
        (6,  'page_view',  'user_004', '2024-01-01 08:25:00'),
        (7,  'click',      'user_005', '2024-01-01 08:30:00')
    ) t(event_id, event_type, user_id, event_time)
""")
conn.close()   # must close before Soda opens the file

# --- Write configuration.yml ---
config_yml = f"""
data_sources:
  analytics_duckdb:
    type: duckdb
    path: "{db_path}"
"""
config_path = tmpdir / "configuration.yml"
config_path.write_text(config_yml)

# --- Run a connectivity check scan (single row_count check) ---
scan = Scan()
scan.set_data_source_name("analytics_duckdb")
scan.add_configuration_yaml_file(str(config_path))
scan.add_sodacl_yaml_str("""
checks for events:
  - row_count > 0
""")
scan.execute()

print("DuckDB connectivity scan:")
print(f"  Connected and ran checks: FAILs={scan.has_check_fails()}, WARNs={scan.has_check_warns()}")
print(scan.get_logs_text())

### What just happened?
- A `row_count > 0` check is the simplest possible **connectivity test** — if it passes, Soda can reach the table.
- We use an absolute path in `configuration.yml` — relative paths can fail depending on the working directory.
- `scan.execute()` opened a new DuckDB connection, ran the SQL, and closed it — the file is now safely released.
- This pattern (one minimal check to validate connectivity) is equivalent to `soda test-connection` in the CLI.

## Step 3 · PostgreSQL Configuration — Reference and Structure

The PostgreSQL configuration follows the same pattern but with more required keys. This is the full reference:

```yaml
data_sources:
  my_postgres:
    type: postgres
    host: localhost         # or RDS endpoint, Cloud SQL IP, etc.
    port: 5432              # default PostgreSQL port
    database: my_database   # database name (NOT schema)
    username: my_user
    password: my_password   # use ${ENV_VAR} in production!
    schema: public          # which schema to inspect by default
    # Optional:
    # sslmode: require       # 'disable', 'allow', 'prefer', 'require', 'verify-ca', 'verify-full'
    # connection_timeout: 30 # seconds
    # query_timeout: 300     # seconds — important for large tables
```

**Key differences from DuckDB:**
- `database` = the PostgreSQL database (like a DuckDB file)
- `schema` = the namespace within the database (like `main` in DuckDB)
- `sslmode: require` is mandatory for most managed PostgreSQL services (RDS, Cloud SQL, Supabase)

In [ ]:
# Generate and validate a PostgreSQL configuration.yml structure
# (We won't connect — no running Postgres available in Colab)
# Production equivalent: soda test-connection -d my_postgres -c configuration.yml

postgres_config = {
    "data_sources": {
        "my_postgres": {
            "type": "postgres",
            "host": "${POSTGRES_HOST}",        # environment variable substitution
            "port": 5432,
            "database": "${POSTGRES_DB}",
            "username": "${POSTGRES_USER}",
            "password": "${POSTGRES_PASSWORD}",
            "schema": "public",
            "sslmode": "require"
        }
    }
}

print("=== PostgreSQL configuration.yml ===")
print(yaml.dump(postgres_config, default_flow_style=False, sort_keys=False))

# Show what the env vars would look like in a .env file (never commit this!)
print("=== Corresponding .env file (add to .gitignore!) ===")
env_example = {
    "POSTGRES_HOST":     "db.example.com",
    "POSTGRES_DB":       "analytics",
    "POSTGRES_USER":     "soda_reader",
    "POSTGRES_PASSWORD": "YOUR_PASSWORD_HERE",  # placeholder — never real credentials
}
for k, v in env_example.items():
    print(f"{k}={v}")

### What just happened?
- **`${VAR_NAME}` syntax** tells Soda to read the value from the process environment at scan time — no secrets in YAML.
- The PostgreSQL user only needs **SELECT privileges** on the tables being scanned — principle of least privilege.
- `sslmode: require` ensures encrypted connections to managed PostgreSQL services.
- The `.env` file pattern keeps credentials local; CI systems inject them as environment secrets.

## Step 4 · Environment Variable Substitution in Depth

Soda uses `${VAR_NAME}` placeholders anywhere in `configuration.yml`. At scan time, Soda calls `os.environ.get('VAR_NAME')` and substitutes the value.

**Three ways to supply environment variables:**

| Method | How | When to use |
|--------|-----|-------------|
| Shell export | `export POSTGRES_HOST=db.example.com` | Local dev, one-off runs |
| `.env` file + `python-dotenv` | `load_dotenv()` before `scan.execute()` | Local dev, team scripts |
| CI secrets | GitHub Actions `env:` block, Airflow Variables | Production pipelines |

**Setting env vars in Python before running a scan:**

```python
import os
os.environ['POSTGRES_HOST'] = 'db.example.com'   # or load from secrets manager
os.environ['POSTGRES_PASSWORD'] = secret_manager.get('postgres-password')
```

In [ ]:
import os

# Demonstrate env var substitution with our DuckDB setup
# Set the db path as an environment variable
os.environ['ANALYTICS_DB_PATH'] = db_path

# Write a configuration.yml that uses ${} substitution
config_with_env_yml = """
data_sources:
  analytics_env:
    type: duckdb
    path: "${ANALYTICS_DB_PATH}"
"""
config_env_path = tmpdir / "configuration_env.yml"
config_env_path.write_text(config_with_env_yml)

print("=== configuration_env.yml (with ${} substitution) ===")
print(config_with_env_yml.strip())
print(f"\nEnvironment variable ANALYTICS_DB_PATH = {os.environ.get('ANALYTICS_DB_PATH')}")

# Run a scan — Soda will resolve ${ANALYTICS_DB_PATH} at runtime
scan_env = Scan()
scan_env.set_data_source_name("analytics_env")
scan_env.add_configuration_yaml_file(str(config_env_path))
scan_env.add_sodacl_yaml_str("""
checks for events:
  - row_count > 0
""")
scan_env.execute()

print("\n=== Scan with env var config ===")
print(f"FAILs: {scan_env.has_check_fails()}, WARNs: {scan_env.has_check_warns()}")
print(scan_env.get_logs_text())

### What just happened?
- `os.environ['ANALYTICS_DB_PATH'] = db_path` injects the value **before** `scan.execute()` — Soda reads it at that point.
- The `configuration.yml` on disk contains `${ANALYTICS_DB_PATH}` — safe to commit because the secret is not in the file.
- **Soda resolves env vars lazily** — the substitution happens when `execute()` reads the config, not when the file is written.
- In a CI pipeline, set secrets in GitHub Actions → `env:` or in Airflow → Variables/Connections.

## Step 5 · Snowflake Configuration — Keys and Concepts

Snowflake has more configuration keys because it has a multi-tier hierarchy:

```
Account → Database → Schema → Table
                  ↕
         Warehouse (compute)
         Role (access control)
```

**Full Snowflake configuration.yml:**

```yaml
data_sources:
  my_snowflake:
    type: snowflake
    username: ${SNOWFLAKE_USER}
    password: ${SNOWFLAKE_PASSWORD}
    account: ${SNOWFLAKE_ACCOUNT}        # e.g. xy12345.us-east-1
    database: MY_DATABASE                 # Snowflake database name (UPPERCASE by default)
    warehouse: MY_WAREHOUSE               # virtual warehouse for compute
    schema: PUBLIC                        # default schema
    role: SODA_READER_ROLE               # role with SELECT on scanned tables
    # Optional:
    # session_parameters:                # Snowflake session overrides
    #   QUERY_TAG: soda-scan
    #   TIMEZONE: UTC
    # connect_args:                      # passed directly to snowflake-connector-python
    #   login_timeout: 30
```

**Snowflake ↔ Soda key mapping:**

| Snowflake concept | Soda config key | Example |
|-------------------|-----------------|--------|
| Account identifier | `account` | `xy12345.us-east-1` |
| Database | `database` | `ANALYTICS` |
| Virtual Warehouse | `warehouse` | `COMPUTE_WH` |
| Role | `role` | `SODA_READER_ROLE` |
| Schema | `schema` | `PUBLIC` or `RAW` |

In [ ]:
# Build a Snowflake configuration as a Python dict and render as YAML
# (No Snowflake connection — this is a reference/documentation cell)

snowflake_config = {
    "data_sources": {
        "my_snowflake": {
            "type": "snowflake",
            "username": "${SNOWFLAKE_USER}",
            "password": "${SNOWFLAKE_PASSWORD}",
            "account":  "${SNOWFLAKE_ACCOUNT}",
            "database": "ANALYTICS",
            "warehouse": "COMPUTE_WH",
            "schema": "PUBLIC",
            "role": "SODA_READER_ROLE",
            "session_parameters": {
                "QUERY_TAG": "soda-scan",
                "TIMEZONE": "UTC"
            }
        }
    }
}

print("=== Snowflake configuration.yml ===")
print(yaml.dump(snowflake_config, default_flow_style=False, sort_keys=False))

# Explain the account format
account_formats = [
    ("Legacy format",          "xy12345"),
    ("Region format",          "xy12345.us-east-1"),
    ("Org + Account format",   "myorg-my_account"),
    ("Private link format",    "xy12345.privatelink"),
]
print("Snowflake account identifier formats:")
for label, fmt in account_formats:
    print(f"  {label:<30} {fmt}")

### What just happened?
- **`account`** is the most confusing Snowflake config key — it's your account identifier, not the full URL.
- `QUERY_TAG` labels Snowflake queries in `QUERY_HISTORY` — essential for cost attribution when multiple tools share a warehouse.
- The `role` key scopes all Soda queries to a specific Snowflake role — create a dedicated `SODA_READER_ROLE` with SELECT-only grants.
- `warehouse` sets the virtual warehouse (compute cluster) — use a small warehouse (XS or S) for Soda scans to minimise cost.

## Step 6 · Connection Validation — Testing Connectivity Programmatically

The Soda CLI offers `soda test-connection -d <source> -c configuration.yml`. With the Python API, the equivalent is running a minimal scan and checking for errors.

We can also **inspect scan logs for connection errors** — Soda logs all connection failures with clear error messages before any checks run.

In [ ]:
def test_connection(data_source_name: str, config_path: str, table_name: str) -> dict:
    """
    Run a minimal Soda scan to test data source connectivity.
    Equivalent to: soda test-connection -d <data_source_name> -c configuration.yml

    Returns a dict with keys:
      connected  (bool): True if scan executed without connection errors
      has_fails  (bool): True if any check failed
      log        (str):  Full scan log text
    """
    from soda.scan import Scan

    scan = Scan()
    scan.set_data_source_name(data_source_name)
    scan.add_configuration_yaml_file(config_path)

    # A single row_count check is the lightest possible query
    scan.add_sodacl_yaml_str(f"""
checks for {table_name}:
  - row_count >= 0
""")

    scan.execute()
    log_text = scan.get_logs_text()

    # Detect connection errors in the log
    error_keywords = ["Error", "Exception", "ConnectionRefused", "OperationalError",
                      "authentication", "timeout", "does not exist"]
    has_error = any(kw.lower() in log_text.lower() for kw in error_keywords)

    return {
        "connected": not has_error,
        "has_fails": scan.has_check_fails(),
        "log": log_text
    }


# Test connectivity to our analytics DuckDB
result = test_connection(
    data_source_name="analytics_duckdb",
    config_path=str(config_path),
    table_name="events"
)

print(f"Connected : {result['connected']}")
print(f"Has fails : {result['has_fails']}")
print("\nLog:")
print(result['log'])

### What just happened?
- `row_count >= 0` always passes on any existing table — it's purely a connectivity probe, not a data quality check.
- We scan the log text for common error keywords — if none found, the connection succeeded.
- **This function is reusable in any pipeline** — wrap it in an Airflow task to gate downstream jobs on connectivity.
- The Soda CLI's `soda test-connection` does essentially the same thing under the hood.

## Step 7 · Multi-Source Configuration — Multiple Sources in One File

A single `configuration.yml` can define multiple data sources. This is useful when your pipeline reads from one source and writes to another — you can scan both in the same project.

In [ ]:
# Create a second DuckDB file to simulate a separate data source
staging_db_path = str(tmpdir / "staging.duckdb")
conn_staging = duckdb.connect(staging_db_path)
conn_staging.execute("""
    CREATE TABLE raw_events AS
    SELECT * FROM (VALUES
        (101, 'impression', 'user_010', '2024-02-01 09:00:00'),
        (102, 'impression', 'user_011', '2024-02-01 09:01:00'),
        (103, 'click',      'user_010', '2024-02-01 09:02:00')
    ) t(event_id, event_type, user_id, event_time)
""")
conn_staging.close()

# Multi-source configuration.yml
multi_config_yml = f"""
data_sources:
  analytics_duckdb:
    type: duckdb
    path: "{db_path}"

  staging_duckdb:
    type: duckdb
    path: "{staging_db_path}"
"""
multi_config_path = tmpdir / "multi_configuration.yml"
multi_config_path.write_text(multi_config_yml)

print("=== Multi-source configuration.yml ===")
print(multi_config_yml.strip())

# Scan the staging source
scan_staging = Scan()
scan_staging.set_data_source_name("staging_duckdb")        # switch to staging source
scan_staging.add_configuration_yaml_file(str(multi_config_path))
scan_staging.add_sodacl_yaml_str("""
checks for raw_events:
  - row_count > 0
  - missing_count(event_id) = 0
""")
scan_staging.execute()

print("\n=== Staging scan results ===")
print(f"FAILs: {scan_staging.has_check_fails()}, WARNs: {scan_staging.has_check_warns()}")
print(scan_staging.get_logs_text())

### What just happened?
- A single `configuration.yml` holds both `analytics_duckdb` and `staging_duckdb` — each scan selects one via `set_data_source_name()`.
- **One configuration file per environment** is the recommended pattern: `configuration_dev.yml`, `configuration_prod.yml`.
- `set_data_source_name()` determines which block in the config is used — it's the selector, not the YAML key.
- You can also build the configuration YAML string dynamically in Python if sources vary at runtime.

In [ ]:
# Challenge: Configure a scan using environment variables
#
# Task:
#   1. Create a new DuckDB file at tmpdir / "challenge.duckdb" with a table:
#      'products' with columns: product_id (int), name (varchar), price (float)
#      Include at least 4 rows.
#   2. Set an environment variable CHALLENGE_DB_PATH pointing to that file.
#   3. Write a configuration.yml string that uses ${CHALLENGE_DB_PATH} for the path.
#   4. Run a scan with these checks on 'products':
#      - row_count > 0
#      - missing_count(product_id) = 0
#      - duplicate_count(product_id) = 0
#   5. Use test_connection() to validate connectivity first,
#      then run the full checks scan.
#
# Expected: connectivity test passes, all checks PASS.

import os

# Your solution here
challenge_db_path = str(tmpdir / "challenge.duckdb")

# TODO: create products table

# TODO: set env var and write configuration

# TODO: call test_connection() then run full scan

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `configuration.yml` | Maps data source names → connection details; lives separate from checks files |
| `type` key | Connector identifier — must match the installed `soda-core-<connector>` package |
| DuckDB config | Only needs `path` — use an absolute file path, not `:memory:` |
| Postgres config | `host`, `port`, `database`, `username`, `password`, `schema`, `sslmode` |
| Snowflake config | `account`, `username`, `password`, `database`, `warehouse`, `schema`, `role` |
| `${VAR}` syntax | Environment variable substitution — resolved at `scan.execute()` time |
| Connectivity test | Run `row_count >= 0` check — if no errors in log, connection succeeded |
| Multi-source config | Multiple `data_sources` blocks in one file; `set_data_source_name()` selects one |

> **Tip:** Always use `${ENV_VAR}` for passwords in `configuration.yml` — commit the file to Git safely while keeping secrets in your CI system or a secrets manager.

---
## What's next
**Day 3** → Write production-ready SodaCL checks — row count, missing values, duplicates, invalid values, and how to organise checks across multiple tables in a single file.

Mark Day 2 complete in your [tracker](../index.html).